# Ứng dụng Canvas

Trong bài học này, bạn sẽ xây dựng một ứng dụng to-do (quản lý công việc), nơi mà AI agent và giao diện người dùng (frontend) chia sẻ chung một trạng thái theo thời gian thực. AI agent có thể tạo/chỉnh sửa các to-do từ backend, và người dùng có thể thao tác trực tiếp trên giao diện (UI) - cả hai phía sẽ tự động được đồng bộ hóa một cách hoàn hảo.

## 📋 Mục tiêu bài học

1. **Trạng thái AI agent chia sẻ** - Giữ cho backend và frontend luôn đồng bộ sau mỗi lượt hội thoại.
2. **Các công cụ backend với khả năng cập nhật trạng thái** - Viết các công cụ có khả năng đọc và cập nhật trạng thái thông qua `Command(update={...})`.
3. **Các công cụ frontend** - Kích hoạt các hành vi trên trình duyệt từ AI agent thông qua hook `useFrontendTool`.

---

## Bước 1: Chuẩn bị môi trường

Đầu tiên, chúng ta cần khởi tạo môi trường, cài đặt các thư viện cần thiết và thiết lập các API key. Trong bài học này, một số hàm tiện ích đã được gói gọn trong file `helper.py` để giúp quá trình thiết lập mượt mà hơn.

In [1]:
# Cài đặt các thư viện phụ thuộc cho frontend
from helper import install_frontend
install_frontend()

# Tải các API key từ biến môi trường
from helper import load_api_keys
load_api_keys()

Installing frontend dependencies ...

up to date, audited 949 packages in 1s

248 packages are looking for funding
  run `npm fund` for details

21 vulnerabilities (2 low, 17 moderate, 2 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm warn allow-scripts 3 packages have install scripts not yet covered by allowScripts:
npm warn allow-scripts   @scarf/scarf@1.4.0 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.28.1 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.25.12 (install: (install scripts present))
npm warn allow-scripts
npm warn allow-scripts Run `npm approve-scripts --allow-scripts-pending` to review, or `npm approve-scripts <pkg>` to allow.
✓ Frontend dependencies installed
✓ OpenAI API key loaded
✓ Google API key loaded


---

## Bước 2: Khởi động backend server

Đoạn code dưới đây sẽ dựng một server FastAPI để host một LangGraph agent tối giản - chưa có tool, chỉ có prompt cơ bản - và kết nối nó với frontend qua AG-UI. 
- `CopilotKitMiddleware` cho phép chia sẻ trạng thái với UI.
- `MemorySaver` lưu trữ trạng thái qua các lượt hội thoại.
- `add_langgraph_fastapi_endpoint` sẽ mount agent tại endpoint `/`. 

*(Chúng ta sẽ mở rộng tool và system_prompt cho agent này ở các bước sau).*

In [2]:
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import CopilotKitMiddleware, LangGraphAGUIAgent
from fastapi import FastAPI
from helper import start_server
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

app = FastAPI()

agent = LangGraphAGUIAgent(
    name="default",
    description="Lesshared-state todo agent",
    graph=create_agent(
        model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"),
        tools=[],
        middleware=[CopilotKitMiddleware()],
        checkpointer=MemorySaver(),
        system_prompt="Bạn là một trợ lý AI hữu ích.",
    ),
)

add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")
start_server(app, port=8006)

✓ Server running at http://localhost:8006


---

## Bước 3: Khởi động frontend

Bạn sẽ xây dựng một bảng panel chứa khung chat và danh sách to-do. AI agent có thể tạo, sửa, xóa to-do, và người dùng cũng có thể tick chọn hoàn thành trực tiếp trên UI.

Hãy khởi động frontend để xem ứng dụng:

In [3]:
from helper import start_frontend, display_app

# Khởi động React frontend
start_frontend(port=3006)

# Hiển thị ứng dụng 
display_app(port=3006)

---

## Bước 4: Trạng thái chia sẻ & Các công cụ backend

**Trạng thái chia sẻ** là một đối tượng JSON có định dạng kiểu dữ liệu rõ ràng mà cả backend agent và frontend đều có thể đọc và ghi. CopilotKit sẽ tự động đồng bộ hóa cả hai bên sau mỗi lượt tương tác.

Bạn sẽ định nghĩa một schema `Todo` và hai công cụ cho agent:
- **`manage_todos`**: Ghi đè toàn bộ danh sách to-do (xử lý việc thêm, sửa, hoặc xóa).
- **`get_todos`**: Đọc danh sách hiện tại để agent có thể kiểm tra trước khi thực hiện thay đổi.

### Định nghĩa schema và tool

*Lưu ý: Chúng ta định nghĩa các object này trực tiếp trong bộ nhớ. Agent server đang chạy có thể tham chiếu trực tiếp tới các cập nhật này mà không cần khởi động lại.*

In [4]:
import uuid
from langchain.agents.middleware import AgentState as BaseAgentState
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from typing_extensions import TypedDict


# Định nghĩa cấu trúc của một Todo
class Todo(TypedDict):
    id: str
    title: str
    completed: bool


# Định nghĩa trạng thái của agent chứa danh sách các Todo
class AgentState(BaseAgentState):
    todos: list[Todo]

@tool
def manage_todos(todos: list[Todo], runtime: ToolRuntime) -> Command:
    """Thay thế toàn bộ danh sách todo. Sử dụng tool này để thêm, sửa, hoặc xóa todos."""
    for todo in todos:
        if not todo.get("id"):
            todo["id"] = str(uuid.uuid4())

    # Trả về command cho phép AI agent cập nhật trạng thái qua tool calls
    return Command(update={
        "todos": todos,
        "messages": [
            ToolMessage(
                content="Cập nhật todos thành công",
                tool_call_id=runtime.tool_call_id,
            )
        ],
    })

@tool
def get_todos(runtime: ToolRuntime):
    """Lấy danh sách todo hiện tại."""
    return runtime.state.get("todos", [])

todo_tools = [manage_todos, get_todos]

### Cập nhật cấu hình agent

Bây giờ, chúng ta sẽ cập nhật `agent.graph` trực tiếp để tích hợp schema và tool vừa tạo, đồng thời hướng dẫn AI bằng một `system_prompt` chi tiết hơn.

In [5]:
agent.graph = create_agent(
    model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite"),
    state_schema=AgentState,
    tools=todo_tools,
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "Bạn quản lý một danh sách công việc chung. "
        "Sử dụng tool manage_todos để thêm, sửa, hoặc xóa các todo. "
        "Sử dụng tool get_todos để kiểm tra danh sách hiện hành. "
        "Khi được yêu cầu quản lý todos, hãy gọi frontend tool 'openOrCloseTodos' với tham số open=true đầu tiên. "
        "Giữ các câu trả lời ngắn gọn trong 1-2 câu."
    ),
)

print("✓ Cập nhật agent graph thành công!")


✓ Cập nhật agent graph thành công!


---

## Bước 5: Cấu hình frontend

Ở phía frontend, chúng ta sẽ sử dụng 2 hook quan trọng từ CopilotKit:
- **`useAgent()`**: Cung cấp kết nối trực tiếp đến trạng thái được chia sẻ của agent. `agent.state.todos` phản ánh danh sách hiện tại và `agent.setState({ todos: updated })` đẩy các thay đổi từ UI ngược lại backend.
- **`useFrontendTool()`**: Đăng ký một công cụ chạy trên trình duyệt. Agent có thể gọi nó như bất kỳ tool backend nào, nhưng logic sẽ được thực thi trên client-side và có thể trực tiếp cập nhật trạng thái React (ví dụ: mở/đóng thanh panel).

Đoạn code sau sẽ lưu nội dung React component vào file `frontend/src/App.tsx`.

In [7]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat, useAgent, useFrontendTool } from "@copilotkit/react-core/v2";
import { useState } from "react";
import { TodoAppLayout } from "@/components/todo-app-layout";
import { TodoList } from "@/components/todo-list";
import { useExampleSuggestions} from "@/hooks/use-example-suggestions";


export default function App() {
  useExampleSuggestions();

  const [todosOpen, setTodosOpen] = useState(false);

  // 🪁 Đăng ký frontend tool để AI agent có thể gọi và điều khiển UI
  useFrontendTool({
    name: "openOrCloseTodos",
    description: "Mở hoặc đóng panel chứa danh sách todo.",
    parameters: z.object({ open: z.boolean()}),
    handler: async ({open}) => {
      setTodosOpen(open);
      return `Panel todos đã được ${ open ? 'mở' : 'đóng'}.`;
    },
  });

  // 🪁 Đăng ký theo dõi trạng thái chung của agent
  const { agent } = useAgent();

  return (
    <TodoAppLayout
      chat={<CopilotChat />}
      open={todosOpen}
      onOpenChange={setTodosOpen}
      panel={(onClose) => (
        <TodoList
          // 🪁 Đọc trạng thái chia sẻ từ backend
          todos={agent.state.todos || []} 

          // 🪁 Ghi/Cập nhật trạng thái chia sẻ từ frontend
          onUpdate={(updated) => agent.setState({ todos: updated })}

          isRunning={agent.isRunning}
          onClose={onClose}
        />
      )}
    />
  );
}

Overwriting frontend/src/App.tsx


---

## Bước 6: Trải nghiệm thực tế!

Hãy làm mới lại ứng dụng (bằng cách chạy lại hàm `display_app` hoặc refresh iframe/trình duyệt) và thử các kịch bản sau trong khung chat:

1. Chat: **"Thêm 3 công việc về việc học CopilotKit"** - Agent sẽ tự động mở panel bằng frontend tool và điền danh sách vào.
2. Tick chọn hoàn thành một công việc **trực tiếp trên giao diện (UI)**.
3. Chat: **"Trong danh sách của tôi hiện đang có những gì?"** - Agent sẽ đọc lại trạng thái mới nhất để trả lời bạn.
4. Chat: **"Xóa tất cả các công việc đã hoàn thành"** - Agent sẽ gọi `manage_todos` với danh sách đã được lọc.

In [ ]:
# Gọi lại hàm display_app để tương tác với UI mới
display_app(port=3006)

---

## 🎯 Tổng kết những gì bạn đã học

- **Trạng thái chia sẻ:** Cung cấp cho cả backend và frontend một nguồn dữ liệu duy nhất - không cần phải tự viết code đồng bộ API thủ công.
- **`Command(update={...})`**: Cho phép công cụ trong LangGraph cập nhật trạng thái đồ thị và trả về kết quả tool trong cùng một bước.
- **`useFrontendTool`**: Phơi bày các hành vi phía trình duyệt (ví dụ như mở panel, điều hướng trang) cho AI agent dưới dạng một công cụ có thể gọi được.
- **`useAgent` + `agent.setState()`**: Cho phép người dùng chỉnh sửa trạng thái trực tiếp từ UI, và AI agent ngay lập tức nhận diện được sự thay đổi đó ở lượt hội thoại tiếp theo.

### Lời kết khóa học

Trải qua lộ trình các bài học, bạn đã đi từ một chatbot cơ bản đến một ứng dụng fullstack AI được đồng bộ hóa hoàn toàn:
- Kết nối LangChain agent với CopilotKit chat UI.
- Đăng ký các component kiểm soát để agent có thể render biểu đồ và card thông tin trực tiếp trong đoạn chat.
- Sử dụng cấu trúc A2UI schema linh hoạt để tách bạch dữ liệu và phần hiển thị.
- Host các MCP app đa năng cho các công cụ như bảng trắng hoặc giao diện thiết kế.
- Đồng bộ trạng thái chia sẻ giữa agent và React app qua các công cụ frontend.

Các kỹ thuật này hoàn toàn có thể kết hợp với nhau. Hãy tiếp tục khám phá tài liệu [CopilotKit Docs](https://docs.copilotkit.ai) và giao thức [AG-UI Protocol](https://docs.ag-ui.com) để xây dựng những ứng dụng vĩ đại hơn nữa!